# Supervised Fine-Tuning of Qwen3-1.7B-Base with Unsloth

Fine-tune [`unsloth/Qwen3-1.7B-Base`](https://huggingface.co/unsloth/Qwen3-1.7B-Base) on a 20k random subset of
[`allenai/tulu-3-sft-mixture`](https://huggingface.co/datasets/allenai/tulu-3-sft-mixture) with LoRA adapters,
using [Unsloth](https://unsloth.ai/) for fast training on a single GPU and [Weights & Biases](https://wandb.ai/)
for experiment tracking.

**Requirements:** a CUDA GPU (set up for Kaggle/Colab). Secrets (`HF_TOKEN`, `wandb-key`) are read from the
host's secret store — never hardcode them.

**Steps:**
1. [Installation](#installation)
2. [Secrets & W&B login](#secrets)
3. [Load the base model](#model)
4. [Add LoRA adapters](#lora)
5. [Chat template](#chat-template)
6. [Data preparation](#data)
7. [Subsample & train/validation split](#split)
8. [Hyperparameters & W&B run](#wandb)
9. [Base-model sanity check](#baseline)
10. [Training](#train)
11. [Inference](#inference)
12. [Saving & export](#save)

Based on the Unsloth notebooks, licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

<a name="installation"></a>
## 1. Installation

Install Unsloth and the pinned `transformers` / `trl` versions this notebook was tested with.
The Colab branch additionally pins `xformers` to the installed torch version; on Kaggle/local,
torch is expected to be preinstalled.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Kaggle / local / cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

<a name="secrets"></a>
## 2. Secrets & W&B login

Read the Hugging Face and Weights & Biases tokens from the host secret store (Kaggle Secrets here)
and log in to W&B so training metrics are tracked.

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
WANDB_TOKEN = user_secrets.get_secret("wandb-key")

In [ ]:
import os
import wandb

os.environ["WANDB_API_KEY"] = WANDB_TOKEN
wandb.login(key=os.environ["WANDB_API_KEY"])

<a name="model"></a>
## 3. Load the base model

Load `Qwen3-1.7B-Base` in 4-bit quantization to reduce memory usage. `dtype = None` lets Unsloth
auto-detect: Float16 for Tesla T4/V100, Bfloat16 for Ampere and newer. RoPE scaling for
`max_seq_length` is handled internally.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None           # auto-detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-1.7B-Base",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

<a name="lora"></a>
## 4. Add LoRA adapters

We add LoRA adapters so we only need to update 1 to 10% of all parameters. With `lora_r = 64` and
`lora_alpha = lora_r` the effective scaling is 1. `use_gradient_checkpointing = "unsloth"` uses 30%
less VRAM and fits 2x larger batch sizes.

In [ ]:
lora_r = 64
lora_alpha = lora_r

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_r,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = lora_alpha,
    lora_dropout = 0,  # 0 is optimized
    bias = "none",     # "none" is optimized
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

<a name="chat-template"></a>
## 5. Chat template

The base model has no instruction format, so we attach Unsloth's `qwen-3` chat template to the
tokenizer. All dataset conversations will later be rendered through this template.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-3",
)

Quick check of the template on a partial assistant turn (`continue_final_message=True` lets the
model continue an already-started assistant message, e.g. to force JSON output):

In [ ]:
chat = [
    {"role": "user", "content": "Can you format the answer in JSON?"},
    {"role": "assistant", "content": '{"name": "'},
]
formatted_chat = tokenizer.apply_chat_template(
    chat,
    tokenize = False,
    continue_final_message = True,
)
print(formatted_chat)

<a name="data"></a>
## 6. Data preparation

We use the [tulu-3-sft-mixture](https://huggingface.co/datasets/allenai/tulu-3-sft-mixture) dataset
(~939k multi-turn conversations in `{"role", "content"}` format) and cache it locally with
`save_to_disk` so re-runs can use `load_from_disk("./tulu_dataset")` instead.

In [ ]:
from datasets import load_dataset, load_from_disk

dataset = load_dataset("allenai/tulu-3-sft-mixture")["train"]

dataset_path = "./tulu_dataset"
dataset.save_to_disk(dataset_path)

Quick look at the dataset and one random example:

In [ ]:
import numpy as np

print(len(dataset))
dataset[np.random.randint(len(dataset))]

### Add a system prompt

Prepend a fixed system prompt to every conversation, so the fine-tuned model learns a consistent
default behavior (concise, honest, refuses harmful requests).

In [ ]:
system_prompt = (
    "You are a helpful, honest AI assistant. Give accurate, concise answers matched to the "
    "question's complexity — say \"I don't know\" rather than guessing. Use a neutral, direct "
    "tone. Refuse illegal or harmful requests with a brief explanation; ask for clarification "
    "when a request is ambiguous"
)
def add_system(example):
    example["messages"] = [{"role": "system", "content": system_prompt}] + example["messages"]
    return example

dataset = dataset.map(add_system)

### Render conversations to text

`standardize_sharegpt` normalizes any ShareGPT-style (`from`/`value`) leftovers into the
`role`/`content` format, then we render each conversation to a single `text` field via the
chat template. SFTTrainer will train on this `text` field.

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return {"text": texts}

In [ ]:
from unsloth.chat_templates import standardize_sharegpt

dataset_formatted = standardize_sharegpt(dataset)
dataset_formatted = dataset_formatted.map(formatting_prompts_func, batched = True)

Sanity check: one fully rendered training example.

In [ ]:
print(dataset_formatted[0]["text"])

<a name="split"></a>
## 7. Subsample & train/validation split

Training on the full ~939k examples would take too long, so we draw a random 20k subset and split
it 80/20 into train and validation sets (seeded for reproducibility).

In [ ]:
N_ELEMENTS = 20000
selected = np.random.randint(0, len(dataset_formatted), N_ELEMENTS)
subset = dataset_formatted.select(selected)

split = subset.train_test_split(test_size = 0.2, seed = 42)
train_data = split["train"]
val_data = split["test"]
print(f"train: {len(train_data)}, val: {len(val_data)}")

<a name="wandb"></a>
## 8. Hyperparameters & W&B run

Define the training hyperparameters up front, then start a timestamped W&B run and log them to the
run config so experiments stay comparable.

In [ ]:
learning_rate = 5e-5
weight_decay = 0.001

MAX_STEPS = 100   # quick run; use num_train_epochs = 1 instead for a full pass
EVAL_STEPS = 10

In [ ]:
import datetime

timestamp = datetime.datetime.today().strftime("%m/%d/%Y - %H:%M")
run = wandb.init(
    project = "Unsloth-FineTuning",
    name = "qwen-3-1.7b-experiment-" + timestamp,
    tags = ["unsloth", "lora", "qwen", "tulu-3-sft"],
)

run.config.update({
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "lora_r": lora_r,
    "lora_alpha": lora_alpha,
    "max_seq_length": max_seq_length,
    "max_steps": MAX_STEPS,
})

<a name="baseline"></a>
## 9. Base-model sanity check

Generate a few answers **before** fine-tuning, so we can compare them with the fine-tuned model
afterwards. The untuned base model is expected to ramble or fail at instruction following.

In [ ]:
prompts = [
    "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,",
    "Resolve this equation 2x+1=0",
    "Write a python function to split a string",
]

for prompt in prompts:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,  # must add for generation
        return_tensors = "pt",
        return_dict = True,
    ).to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True,
                             temperature = 0.7, min_p = 0.1)
    print(tokenizer.batch_decode(outputs)[0])
    print("-" * 80)

<a name="train"></a>
## 10. Training

We train with TRL's `SFTTrainer` for `max_steps = 100` to keep the run short — for a full run set
`num_train_epochs = 1` and `max_steps = None`. Evaluation runs every `EVAL_STEPS` steps on the
validation split, and metrics are reported to W&B.

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    eval_dataset = val_data,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = MAX_STEPS,
        learning_rate = learning_rate,
        eval_strategy = "steps",
        eval_steps = EVAL_STEPS,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = weight_decay,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
    ),
)

We also use Unsloth's `train_on_responses_only` to compute loss only on the assistant outputs and
ignore the user's inputs. Unsloth auto-detects the instruction/response parts from the tokenizer's
chat template, so no explicit markers are needed.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer)

Verify the masking: decoding the labels with `-100` (masked) replaced by a space should show only
the assistant responses — system and user turns are masked out.

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

GPU memory before training, for reference:

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

Final memory and time stats:

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="inference"></a>
## 11. Inference

Run the fine-tuned model on the same kind of prompts used in the sanity check. `for_inference`
enables Unsloth's native 2x faster inference. We sample with `min_p = 0.1` and
`temperature = 0.7` (see this [tweet](https://x.com/menhguin/status/1826132708508213629) for why
min-p sampling).

In [ ]:
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True,
                         temperature = 0.7, min_p = 0.1)
tokenizer.batch_decode(outputs)

Use a `TextStreamer` to see the generation token by token instead of waiting for the full
completion:

In [ ]:
from transformers import TextStreamer

messages = [{"role": "user", "content": "Resolve this equation 2x+1=0"}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 0.7, min_p = 0.1)

<a name="save"></a>
## 12. Saving & export

Merge the LoRA adapters into the base model and save a 16-bit version locally (suitable for vLLM).
The commented lines show how to push to the Hub, or how to save only the small LoRA adapters
instead.

In [ ]:
# Merge LoRA adapters into a 16-bit model and save locally
model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit")

# Upload the merged model to the Hub
# model.push_to_hub_merged("FrancescoArno94/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = HF_TOKEN)

# Save only the LoRA adapters instead
# model.save_pretrained("qwen_lora")
# tokenizer.save_pretrained("qwen_lora")
# model.push_to_hub("HF_USERNAME/qwen_lora", token = HF_TOKEN)
# tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token = HF_TOKEN)

To reload previously saved LoRA adapters for inference, set `if False` to `if True`:

In [ ]:
if False:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_lora",  # the adapters saved above
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "Describe a tall tower in the capital of France."}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 0.7, min_p = 0.1)

### Optional: GGUF export

For llama.cpp / Ollama-style deployment, Unsloth can convert directly to GGUF. Supported quants
include `q8_0` (fast conversion), `q4_k_m` and `q5_k_m` (recommended) — see the
[docs](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf) for the full list.
Everything is commented out since it is disabled by default.

In [ ]:
# Save to GGUF locally (default q8_0)
# model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")

# Export multiple quants and upload to the Hub in one call
# model.push_to_hub_gguf(
#     "HF_USERNAME/qwen_finetune",
#     tokenizer,
#     quantization_method = ["q4_k_m", "q8_0", "q5_k_m"],
#     token = HF_TOKEN,
# )